In [1]:
from core.model_providers.openai_llm import OpenAILLM
from core import KAGConfig
import json
import re


config = KAGConfig.from_yaml("configs/config_openai.yaml")
llm = OpenAILLM(config)


with open("data/knowledge_graph/doc2chunks.json", "r") as f:
    desc_chunks = json.load(f)

def clean_screenplay_text(content: str) -> str:
    s = content

    # 1) 修复断行导致的拆词：word-\nword  -> wordword
    #    只处理连字符紧贴在行尾的情况（最安全）
    s = re.sub(r'([A-Za-z])-\s*\n\s*([A-Za-z])', r'\1\2', s)

    # 2) 可选：修复你这个例子里已经变成 "bur- ied" 这种形式的拆词
    #    这一步更激进，建议只在你确认数据里大量存在该模式时启用
    s = re.sub(r'([A-Za-z])-\s+([a-z])', r'\1\2', s)

    # 3) 最后再压缩空白
    s = re.sub(r'\s+', ' ', s).strip()
    return s

/vepfs-mlp2/c20250513/241404044/users/roytian/anaconda3/envs/screenplay/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
desc_chunks

{'scene_1_part_1': {'document_metadata': {'doc_title': '1 INT.  COMMUNITY HALL.   DAY.',
   'title': '1 INT.  COMMUNITY HALL.   DAY.',
   'subtitle': '',
   'order': 0,
   'version': 'default',
   'scene_id': '1',
   'scene_category': 'INT',
   'lighting': 'Day',
   'space': 'real world',
   'region': None,
   'main_location': 'Community Hall',
   'sub_location': None,
   'summary': 'John embraces a 600-mile journey as a positive change, feeling empowered and grateful to his fellowship for supporting his path to freedom, not escape.',
   'source_doc_id': 'doc_0',
   'source_title': '1 INT.  COMMUNITY HALL.   DAY.'},
  'chunks': [{'id': 'doc_0_chunk_0',
    'content': 'The Church of St. Peter Los Angeles. "WHOEVER YOU SEE HERE - WHATEVER YOU HEAR HERE - STAYS HERE." That\'s a notice on a wall. Here\'s another notice "NO SMOKING." Everyone is smok- ing. This is an AA meeting. There\'s a lot of Faces to look at. I don\'t know when we\'ll get to the one that\'s talking, but when we do it\'

In [3]:
content = desc_chunks["scene_1_part_1"]["chunks"][0]["content"]
cleaned = clean_screenplay_text(content)
print(cleaned)

The Church of St. Peter Los Angeles. "WHOEVER YOU SEE HERE - WHATEVER YOU HEAR HERE - STAYS HERE." That's a notice on a wall. Here's another notice "NO SMOKING." Everyone is smoking. This is an AA meeting. There's a lot of Faces to look at. I don't know when we'll get to the one that's talking, but when we do it's like this. Eyes like glue. 50 years old with a face the color of a snuff-users hanky. He says this: BENNY .. after my third recovery my wife made me swear I'd never bring another bottle into the house. And I never did. I buried it under the lawn. Cut out a turf & stood it upright with a piece of tinfoil instead of a cork. So here we are out in the yard, and she's happy because I'm getting healthy in a pair of swiming shorts & no way near no booze. She decides to prune the roses. Meanwhile, I'm laying there with a straw stuck into the fucken lawn doing a quart of red .. Curious thing about drunks. Their disease often amuses them. That's how crazy I was - I was sick for half a 

In [4]:
import json
from enum import Enum
from langgraph.graph import StateGraph, END
from core.utils.format import correct_json_format
from typing import List, Dict, Any, Tuple, Set
from core.builder.manager.information_manager import InformationExtractor
import asyncio  
import os
import logging

extractor = InformationExtractor(config, llm)

In [5]:
entity_extraction = []

for entity_type in ["event", "time_and_location", "general"]:
    existing = [f"entity name: {ent["name"]}        entity type {ent["type"]}" for ent in entity_extraction]
    result = json.loads(extractor.extract_entities(text=cleaned, entity_type=entity_type, extracted_entities="\n".join(existing)))
    if entity_type.lower() == "event":
        for res in result:
            if res["type"] == "Event":
                res["scope"] = "local"
            elif res["type"] == "Occasion":
                res["scope"] = "global"
            else: # failure
                continue
    entity_extraction.extend(result)

TypeError: InformationExtractor.extract_entities() got an unexpected keyword argument 'entity_type'

In [ ]:
entity_extraction

[{'type': 'Occasion',
  'name': 'Alcoholics Anonymous meeting at Church of St. Peter Los Angeles',
  'description': 'A local AA meeting taking place in the Church of St. Peter in Los Angeles, where individuals gather to share personal stories and support each other in recovery from alcoholism.',
  'scope': 'global'},
 {'type': 'Event',
  'name': 'Benny shares his recovery story at the AA meeting',
  'description': 'Benny recounts his experience of hiding alcohol under the lawn turf and drinking through a straw while pretending to be sober, reflecting on the absurdity of his addiction and his eventual path to recovery.',
  'scope': 'local'},
 {'type': 'Event',
  'name': "John Berlin is introduced after Benny's story",
  'description': 'After Benny finishes speaking, attention shifts to John Berlin, a man in his forties with intense eyes and dark hair, who is now being acknowledged in the meeting.',
  'scope': 'local'},
 {'type': 'Location',
  'name': 'Church of St. Peter, Los Angeles',


In [ ]:
with open("../NarrativeWeaver/core/schema/default_relation_schema.json", "r") as f:
    relation_schema = json.load(f)

with open("../NarrativeWeaver/core/schema/default_entity_schema.json", "r") as f:
    entity_schema = json.load(f)

In [ ]:
def generate_schema_text(group):
    lines = []
    for rel in group:
        if rel["direction"] == "directed":
            symbol = "-->" 
        else:
            symbol = "<-->"

        rule = "/".join(rel["from"]) + symbol + "/".join(rel["to"])
        line = f"{rel['type']}: {rel['description']}         constraint: {rule}" 
        lines.append(line)

    return "\n".join(lines)

def prepare_few_shot_examples(group):
    samples = []
    for rel in group:
        samples.extend(rel["samples"])

    return json.dumps(samples, ensure_ascii=False, indent=2)#.replace("{", "{{").replace("}", "}}")

In [ ]:
import re
from typing import Dict, Iterable, Tuple

_whitespace_re = re.compile(r"\s+")

def normalize_name_basic(name: str) -> str:
    """
    Basic normalization for any entity name:
    - remove leading/trailing whitespace
    - convert any whitespace (tabs/newlines/multiple spaces) into a single space
    """
    if name is None:
        return ""
    name = str(name)
    name = _whitespace_re.sub(" ", name).strip()
    return name

def titlecase_global_name(name: str) -> str:
    """
    For global-scope names: title-case each word.
    Example: 'brandon   roy' -> 'Brandon Roy'
    Keeps punctuation reasonably (e.g., "O'Neil" -> "O'Neil" via .title()).
    """
    name = normalize_name_basic(name)
    # Python's .title() is usually fine for person/place names; if you have many edge cases, we can refine later.
    return name.title()

def canonicalize_entity_name(name: str, scope: str) -> str:
    name = normalize_name_basic(name)
    if scope == "global":
        name = titlecase_global_name(name)
    return name

def build_name_canonicalizer(entities: Iterable[Dict]) -> Tuple[Dict[str, str], Dict[str, str]]:
    """
    Returns:
    - raw2canon: map from raw name to canonical name (only for names seen in entities)
    - canon2raw: reverse map (last one wins) if you need debugging
    """
    raw2canon = {}
    canon2raw = {}
    for ent in entities:
        raw = ent.get("name", "")
        scope = ent.get("scope", "local")
        canon = canonicalize_entity_name(raw, scope)
        raw2canon[raw] = canon
        canon2raw[canon] = raw
    return raw2canon, canon2raw

def apply_canonicalization_to_entities(entities: Iterable[Dict]) -> list:
    out = []
    for ent in entities:
        ent = dict(ent)
        ent["name"] = canonicalize_entity_name(ent.get("name", ""), ent.get("scope", "local"))
        out.append(ent)
    return out

def apply_canonicalization_to_relations(relations: Iterable[Dict], raw2canon: Dict[str, str]) -> list:
    """
    Canonicalize subject/object by lookup.
    Also applies basic normalization even if not found in map.
    """
    out = []
    for rel in relations:
        rel = dict(rel)
        subj_raw = rel.get("subject", "")
        obj_raw = rel.get("object", "")
        rel["subject"] = raw2canon.get(subj_raw, normalize_name_basic(subj_raw))
        rel["object"] = raw2canon.get(obj_raw, normalize_name_basic(obj_raw))
        out.append(rel)
    return out


In [ ]:
entity_extraction = apply_canonicalization_to_entities(entity_extraction)
raw2canon, _ = build_name_canonicalizer(entity_extraction)

entity_name_selector = {ent["name"] for ent in entity_extraction}
entity_name2type = {ent["name"]: ent["type"] for ent in entity_extraction}

In [ ]:
import json
from typing import Any, Dict, List, Tuple, Callable, Optional


# -----------------------------
# Helpers: context building
# -----------------------------
def build_entities_text_for_group(
    entity_extraction: List[Dict[str, Any]],
    group: List[Dict[str, Any]],
) -> str:
    """Build entity list text filtered by types that appear in this relation group schema."""
    entity_type_selector = {t for rel in group for t in (rel["from"] + rel["to"])}
    lines = [
        f'entity_name: {ent["name"]}        type {ent["type"]}'
        for ent in entity_extraction
        if ent.get("type") in entity_type_selector
    ]
    return "\n".join(lines)


# -----------------------------
# Step 1: LLM extraction
# -----------------------------
def extract_relations_llm(
    *,
    content: str,
    entity_extraction: List[Dict[str, Any]],
    extracted_relations: List[Dict[str, Any]],
    group: List[Dict[str, Any]],
    previous_results: str = None,
    feedbacks: str = None
) -> List[Dict[str, Any]]:
    """
    LLM extraction only.
    Returns raw relations parsed from JSON (no rid, no validation).
    """
    entities_text = build_entities_text_for_group(entity_extraction, group)
    relation_schema_text = generate_schema_text(group)
    few_shot_samples = prepare_few_shot_examples(group)

    raw = extractor.extract_relations_enhanced(
        text=content,
        extracted_entities=entities_text,
        extracted_relations=extracted_relations,
        relation_schema_text=relation_schema_text,
        relation_few_shot_examples=few_shot_samples,
        previous_results=previous_results,
        feedbacks=feedbacks
    )
    return json.loads(raw)


# -----------------------------
# Step 2: postprocess (canonicalize + rid)
# -----------------------------
def assign_rids(relations: List[Dict[str, Any]], rid_prefix: str) -> None:
    """Assign global-unique rid with a group-specific prefix."""
    for i, rel in enumerate(relations, start=1):
        rel["rid"] = f"{rid_prefix}#{i}"


def postprocess_relations(
    relations: List[Dict[str, Any]],
    *,
    rid_prefix: str,
    apply_canonicalization_to_relations: Callable[[List[Dict[str, Any]], Dict[str, str]], List[Dict[str, Any]]],
    raw2canon: Dict[str, str],
) -> List[Dict[str, Any]]:
    """Canonicalize and assign rids."""
    relations = apply_canonicalization_to_relations(relations, raw2canon)
    assign_rids(relations, rid_prefix)
    return relations


# -----------------------------
# Step 3: validation + feedback
# -----------------------------
def build_validation_context(
    entity_extraction: List[Dict[str, Any]],
    group: List[Dict[str, Any]],
) -> Dict[str, Any]:


    relation_type_selector = {r["type"] for r in group}
    rule_finder = {
        r["type"]: {"from": r["from"], "to": r["to"], "direction": r["direction"]}
        for r in group
    }

    return {
        "relation_type_selector": relation_type_selector,
        "rule_finder": rule_finder,
    }


def add_feedback(
    feedbacks: Dict[str, List[Dict[str, Any]]],
    *,
    key: str,
    rid: str,
    feedback: str,
    subj: str,
    obj: str,
    rtype: str,
) -> None:
    payload = {
        "rid": rid,
        "feedback": feedback,
        "subject": subj,
        "object": obj,
        "relation_type": rtype,
    }
    feedbacks.setdefault(key, []).append(payload)


def validate_and_fix_relations(
    relations: List[Dict[str, Any]],
    *,
    entity_extraction: List[Dict[str, Any]],
    group: List[Dict[str, Any]],
) -> Tuple[List[Dict[str, Any]], Dict[str, List[Dict[str, Any]]]]:
    """
    Rule-based checks and minimal auto-fix (swap for directed relation if it satisfies type constraint).
    Sets rel["conf"] and possibly rel["auto_fixed"]/rel["fix_reason"].
    """
    ctx = build_validation_context(entity_extraction, group)

    relation_type_selector = ctx["relation_type_selector"]
    rule_finder = ctx["rule_finder"]

    feedbacks: Dict[str, List[Dict[str, Any]]] = {}

    for rel in relations:
        rid = rel.get("rid", "")
        subj = rel.get("subject", "")
        obj = rel.get("object", "")
        rtype = rel.get("relation_type", "")
        rname = rel.get("relation_name", "")

        # subject existence
        if subj not in entity_name_selector:
            add_feedback(
                feedbacks,
                key="subject error",
                rid=rid,
                feedback=(
                    f"Subject [{subj}] is not previously extracted. "
                    f"Consider adding the entity or dropping the relation."
                ),
                subj=subj,
                obj=obj,
                rtype=rtype,
            )
            rel["conf"] = 0.0
            continue

        # object existence
        if obj not in entity_name_selector:
            add_feedback(
                feedbacks,
                key="object error",
                rid=rid,
                feedback=(
                    f"Object [{obj}] is not previously extracted. "
                    f"Consider adding the entity or dropping the relation."
                ),
                subj=subj,
                obj=obj,
                rtype=rtype,
            )
            rel["conf"] = 0.0
            continue

        # relation type existence
        if rtype not in relation_type_selector:
            add_feedback(
                feedbacks,
                key="undefined relation error",
                rid=rid,
                feedback=(
                    f"Relation type [{rtype}] (name [{rname}]) is not defined in schema. "
                    f"Consider modifying or dropping the relation."
                ),
                subj=subj,
                obj=obj,
                rtype=rtype,
            )
            rel["conf"] = 0.0
            continue

        # type lookup
        subject_type = entity_name2type.get(subj)
        if subject_type is None:
            add_feedback(
                feedbacks,
                key="subject error",
                rid=rid,
                feedback=(
                    f"Subject [{subj}] exists but its type is missing. "
                    f"Fix entity typing or drop the relation."
                ),
                subj=subj,
                obj=obj,
                rtype=rtype,
            )
            rel["conf"] = 0.0
            continue

        object_type = entity_name2type.get(obj)
        if object_type is None:
            add_feedback(
                feedbacks,
                key="object error",
                rid=rid,
                feedback=(
                    f"Object [{obj}] exists but its type is missing. "
                    f"Fix entity typing or drop the relation."
                ),
                subj=subj,
                obj=obj,
                rtype=rtype,
            )
            rel["conf"] = 0.0
            continue

        # rule lookup
        rule = rule_finder.get(rtype)
        if rule is None:
            add_feedback(
                feedbacks,
                key="missing rule error",
                rid=rid,
                feedback=f"No rule defined for relation type [{rtype}].",
                subj=subj,
                obj=obj,
                rtype=rtype,
            )
            rel["conf"] = 0.0
            continue

        direction = rule.get("direction")

        # rule satisfaction
        if direction == "symmetric":
            rule_satisfied = (
                (subject_type in rule["from"] and object_type in rule["to"])
                or (subject_type in rule["to"] and object_type in rule["from"])
            )
        else:
            rule_satisfied = (subject_type in rule["from"] and object_type in rule["to"])

        if not rule_satisfied:
            if direction == "directed":
                flipped_satisfied = (object_type in rule["from"] and subject_type in rule["to"])
                if flipped_satisfied:
                    rel["subject"], rel["object"] = obj, subj
                    rel["auto_fixed"] = True
                    rel["fix_reason"] = "swap_subject_object_to_satisfy_type_constraint"
                    rel["conf"] = 0.5
                    continue

            add_feedback(
                feedbacks,
                key="constraint violation error",
                rid=rid,
                feedback=(
                    f"Type constraint violated: subject type [{subject_type}] and object type [{object_type}] "
                    f"do not satisfy {rule['from']} -> {rule['to']} for the relation type {rel["relation_type"]}."
                ),
                subj=subj,
                obj=obj,
                rtype=rtype,
            )
            rel["conf"] = 0.0
        else:
            rel["conf"] = 0.8

    return relations, feedbacks


# -----------------------------
# Main wrapper: short and clear
# -----------------------------
def extract_relations_by_ground(
    entity_extraction: List[Dict[str, Any]],
    group: List[Dict[str, Any]],
    content: str,
    *,
    rid_prefix: str,
    extractor: Any,
    generate_schema_text: Callable[[List[Dict[str, Any]]], str],
    prepare_few_shot_examples: Callable[[List[Dict[str, Any]]], str],
    apply_canonicalization_to_relations: Callable[[List[Dict[str, Any]], Dict[str, str]], List[Dict[str, Any]]],
    raw2canon: Dict[str, str],
) -> Tuple[List[Dict[str, Any]], Dict[str, List[Dict[str, Any]]]]:
    """
    Wrapper: LLM extraction -> canonicalize+rid -> validate+feedback.
    """
    relations = extract_relations_llm(
        content=content,
        entity_extraction=entity_extraction,
        group=group,
        extractor=extractor,
        generate_schema_text=generate_schema_text,
        prepare_few_shot_examples=prepare_few_shot_examples,
    )
    relations = postprocess_relations(
        relations,
        rid_prefix=rid_prefix,
        apply_canonicalization_to_relations=apply_canonicalization_to_relations,
        raw2canon=raw2canon,
    )
    return validate_and_fix_relations(
        relations,
        entity_extraction=entity_extraction,
        group=group,
    )


def filter_illegal_is_a_relations(
    relations: list[dict],
    *,
    entity_extraction: list[dict],
    entity_type_names: set[str] | None = None,
) -> list[dict]:
    """
    Hard filter for illegal `is_a` relations:
      - subject must be an existing entity name
      - subject must NOT be a schema type name (Event/Occasion/Location/TimePoint/Character/Object/Concept, etc.)
      - object must NOT be a schema type name
    If violated, drop the relation.
    """
    if not relations:
        return relations

    if entity_type_names is None:
        entity_type_names = {
            "Event", "Occasion", "Location", "TimePoint",
            "Character", "Object", "Concept", "Action"
        }

    # normalize to compare safely
    type_name_norm = {t.strip().lower() for t in entity_type_names}

    entity_names = {e.get("name", "").strip() for e in (entity_extraction or []) if e.get("name")}
    entity_names_norm = {n.lower() for n in entity_names}

    kept = []
    for rel in relations:
        rtype = (rel.get("relation_type") or rel.get("type") or "").strip()
        if rtype != "is_a":
            kept.append(rel)
            continue

        subj = (rel.get("subject") or "").strip()
        obj = (rel.get("object") or "").strip()

        subj_norm = subj.lower()
        obj_norm = obj.lower()

        # subject must be a previously extracted entity name
        if subj_norm not in entity_names_norm:
            continue

        # subject/object must not be schema type names
        if subj_norm in type_name_norm:
            continue
        if obj_norm in type_name_norm:
            continue

        kept.append(rel)

    return kept


In [ ]:
from tqdm import tqdm

all_relations = {}   # rid -> relation dict
all_feedbacks = {}   # error_key -> list[feedback dict]
all_extracted_relations = []

for relation_group in tqdm(relation_schema):
    group = relation_schema[relation_group]

    relation_extraction = extract_relations_llm(
        content=content,
        entity_extraction=entity_extraction,
        group=group,
        extracted_relations=json.dumps(all_extracted_relations, ensure_ascii=False, indent=2) or ""
    )
    all_extracted_relations.extend(relation_extraction)
    
    relation_extraction = filter_illegal_is_a_relations(
        relation_extraction,
        entity_extraction=entity_extraction,
    )
    relation_extraction = postprocess_relations(
        relation_extraction,
        rid_prefix=relation_group,
        apply_canonicalization_to_relations=apply_canonicalization_to_relations,
        raw2canon=raw2canon,
    )

    relation_extraction, feedbacks = validate_and_fix_relations(
        relation_extraction,
        entity_extraction=entity_extraction,
        group=group,
    )


    # 1) 累积关系：dict by rid
    for rel in (relation_extraction or []):
        rid = rel.get("rid")
        if not rid:
            raise ValueError(f"Missing rid in relation: {rel}")
        if rid in all_relations:
            raise ValueError(f"Duplicate rid detected: {rid}")
        all_relations[rid] = rel

    # 2) 累积反馈：按 key 合并（rid 已经天然一致）
    if feedbacks:
        for k, items in feedbacks.items():
            all_feedbacks.setdefault(k, [])
            if isinstance(items, list):
                all_feedbacks[k].extend(items)
            else:
                all_feedbacks[k].append(items)

# 3) 一致性校验：确保每条 feedback 的 rid 都能定位到关系
missing_rids = []
for k, items in all_feedbacks.items():
    for fb in items:
        rid = fb.get("rid")
        if rid not in all_relations:
            missing_rids.append((k, rid, fb.get("message", "")))

if missing_rids:
    # 打印前几个帮助 debug
    preview = "\n".join([f"{k}: {rid} -> {msg}" for k, rid, msg in missing_rids[:10]])
    raise ValueError(
        f"Found feedbacks pointing to unknown rids. Count={len(missing_rids)}\n{preview}"
    )

  0%|          | 0/7 [00:00<?, ?it/s]

100%|██████████| 7/7 [01:37<00:00, 13.86s/it]


In [ ]:
all_feedbacks

{'object error': [{'rid': 'object_relations#4',
   'feedback': 'Object [notes] is not previously extracted. Consider adding the entity or dropping the relation.',
   'subject': 'St Anne',
   'object': 'notes',
   'relation_type': 'possesses'}],
 'constraint violation error': [{'rid': 'spatiotemporal_relations#15',
   'feedback': "Type constraint violated: subject type [Event] and object type [TimePoint] do not satisfy ['Event'] -> ['Occasion'] for the relation type occurs_during.",
   'subject': "St ANNE asks BERLIN about the janitor's presence at the institute",
   'object': 'that night',
   'relation_type': 'occurs_during'},
  {'rid': 'semantic_relations#1',
   'feedback': "Type constraint violated: subject type [Event] and object type [Object] do not satisfy ['Location', 'Object', 'Concept', 'Occasion'] -> ['Location', 'Object', 'Concept', 'Occasion'] for the relation type part_of.",
   'subject': 'St ANNE offers BERLIN a cigarette',
   'object': 'cigarette',
   'relation_type': 'pa

In [ ]:
from collections import defaultdict
from typing import Dict, Set, Tuple, Any

def build_typepair_relation_index(
    global_rule_finder: Dict[str, Dict[str, Any]]
) -> Dict[Tuple[str, str], Set[str]]:
    """
    Build a simplified index:
      (type_a, type_b) -> {relation_type, ...}

    - Ignores direction
    - Treats (A, B) and (B, A) as the same key
    - Deduplicates relation types
    """
    index = defaultdict(set)

    for relation_type, rule in global_rule_finder.items():
        from_types = rule.get("from", [])
        to_types = rule.get("to", [])

        # 所有可能的类型组合（无向）
        for t1 in from_types:
            for t2 in to_types:
                key = tuple(sorted((t1, t2)))
                index[key].add(relation_type)

    return dict(index)

def get_allowed_relations_between_types(
    typepair_index: Dict[Tuple[str, str], List[Dict[str, Any]]],
    subject_type: str,
    object_type: str,
) -> List[Dict[str, Any]]:
    """
    Return allowed relations if subject_type -> object_type is valid.
    """
    return typepair_index.get((subject_type, object_type), [])


relation_type_info = {}

for group in relation_schema.values():   # 如果 relation_schema 是 dict
    for r in group:
        rtype = r["type"]
        if rtype in relation_type_info:
            raise ValueError(
                f"Duplicate relation_type detected across groups: {rtype}"
            )
        relation_type_info[rtype] = {
            "from": r["from"],
            "to": r["to"],
            "direction": r["direction"],
            "description": r["description"]
        }


typepair_index = build_typepair_relation_index(relation_type_info)

In [ ]:
def get_allowed_list(allowed_forward, allowed_backward):
    if allowed_forward:
        outputs = [f"{rel}: {relation_type_info.get(rel)["description"]}" for rel in allowed_forward if relation_type_info.get(rel)]
        return "\n".join(outputs), "select"
    else:
        if allowed_backward:
            outputs = [f"{rel}: {relation_type_info.get(rel)["description"]}" for rel in allowed_backward if relation_type_info.get(rel)]
            final_text = "\n".join(outputs) + "\nPlease consider reverse the subject and object in extracted relation to match the above relation type(s)"
            return final_text, "reverse_select"
        else:
            return None, "drop"

In [ ]:
def fix_relation_error(error, content):
    rid = error["rid"]
    extracted_relation = all_relations[rid]
    feedback = error["feedback"]
    subject = error["subject"]
    subject_type = entity_name2type[subject]
    object = error["object"]
    object_type =  entity_name2type[object]

    allowed_forward = get_allowed_relations_between_types(typepair_index, subject_type, object_type)
    allowed_backward = get_allowed_relations_between_types(typepair_index, object_type, subject_type)
    allowed_list, action = get_allowed_list(allowed_forward, allowed_backward)
    if action == "drop":
        return "drop", {} 
    result = extractor.fix_relation_error(text=content, extracted_relation=json.dumps(extracted_relation, ensure_ascii=False, indent=2),
                                          allowed_relation_types=allowed_list, feedback=feedback)
    result = json.loads(result)
    decision = result["decision"]
    output = result["output"]
    if action == "select":
        return decision, output
    if action == "reverse_select":
        if decision == "rewrite":
            output["subject"] = object
            output["object"] = subject
            return decision, output
    return "drop", {}

In [ ]:
entity_type_description = {ent["type"]: ent["description"] for ent in entity_schema}

def fix_entity_error(error, error_type, content):
    rid = error["rid"]
    # extracted_relation = all_relations[rid]
    feedback = error["feedback"]
    subject = error["subject"]
    # subject_type = entity_name2type[subject]
    object = error["object"]
    # object_type =  entity_name2type[object]
    relation_type = error["relation_type"]
    if error_type == "subject error":
        candidate_list = relation_type_info[relation_type]["from"] 
    else:
        candidate_list = relation_type_info[relation_type]["to"] 
    entity_type_description_text = "\n".join([f"{ent}: {entity_type_description[ent]}" for ent in candidate_list])
    result = extractor.fix_entity_error(text=content, candidate_entity_types=", ".join(candidate_list), candidate_entity_descriptions=entity_type_description_text, feedback=feedback)
    result = json.loads(result)
    decision = result["decision"]
    output = result["output"]
    output["subject"] = object
    output["object"] = subject
    return decision, output

In [ ]:
def resolve_errors(entity_extraction, all_relations, all_feedbacks, content):
    # 遍历 key 的快照，避免迭代时 dict 结构变化带来不确定行为
    for error_type in list(all_feedbacks.keys()):
        error_list = all_feedbacks.get(error_type, [])
        if not error_list:
            continue

        idx = 0
        while idx < len(error_list):
            error = error_list[idx]
            rid = error.get("rid")

            # 防御：rid 已经不存在就把这个 feedback 丢掉
            if not rid or rid not in all_relations:
                error_list.pop(idx)
                continue

            if error_type in ["undefined relation error", "missing rule error", "constraint violation error"]:
                decision, output = fix_relation_error(error, content)

                if decision == "drop":
                    # drop relation
                    all_relations.pop(rid, None)
                    # drop feedback
                    error_list.pop(idx)
                    continue
                else:
                    # update relation
                    # 保险：保留 rid，避免 fix 输出没带 rid
                    output = dict(output)
                    output.setdefault("rid", rid)
                    all_relations[rid] = output
                    # drop feedback
                    error_list.pop(idx)
                    continue

            elif error_type in ["subject error", "object error"]:
                decision, output = fix_entity_error(error, error_type, content)

                if decision == "drop":
                    all_relations.pop(rid, None)
                    error_list.pop(idx)
                    continue
                else:
                    # add entity
                    entity_extraction.append(output)
                    # drop feedback
                    error_list.pop(idx)
                    continue

            else:
                # 未识别的错误类型：你现在策略是直接 drop
                all_relations.pop(rid, None)
                error_list.pop(idx)
                continue


    return entity_extraction, all_relations, all_feedbacks


In [ ]:
entity_extraction, all_relations, all_feedbacks = resolve_errors(entity_extraction, all_relations, all_feedbacks, cleaned)

In [ ]:
all_feedbacks 

{'object error': [], 'constraint violation error': []}

In [ ]:
import json
from collections import defaultdict

def dedup_multi_relations(
    all_relations: dict[str, dict],
    content: str,
) -> tuple[list[dict], dict[str, dict]]:
    """
    Deduplicate relations between the same entity pair.

    Rules:
      1) If relation_type is the same for the same pair -> keep the first, drop the rest.
      2) If relation_type differs for the same pair -> ask LLM (RelationDeduper):
           - decision="keep": do nothing
           - decision="drop": drop all relations whose relation_type in output.drop_relation_types

    Returns:
      all_relations
    """
    # --- build buckets: pair -> list of rids (preserve insertion order) ---
    buckets = defaultdict(list)
    for rid, rel in all_relations.items():
        if not isinstance(rel, dict):
            continue
        s = (rel.get("subject") or "").strip()
        o = (rel.get("object") or "").strip()
        if not s or not o:
            continue

        key = (s, o)
        buckets[key].append(rid)

    # --- iterate each pair that has multiple relations ---
    for pair, rids in list(buckets.items()):
        if len(rids) < 2:
            continue

        # Pull relations in the same order
        rels = []
        for rid in rids:
            rel = all_relations.get(rid)
            if isinstance(rel, dict):
                rels.append({**rel, "rid": rid})

        if len(rels) < 2:
            continue

        # -----------------------
        # Rule 1: same type keep first
        # -----------------------
        seen_types = set()
        drop_rids_same_type = []
        kept_rels = []

        for rel in rels:
            rid = rel.get("rid")
            rtype = (rel.get("relation_type") or rel.get("type") or "").strip()

            # If missing rid or type, keep it (conservative)
            if not rid or not rtype:
                kept_rels.append(rel)
                continue

            if rtype in seen_types:
                drop_rids_same_type.append(rid)
            else:
                seen_types.add(rtype)
                kept_rels.append(rel)

        for rid in drop_rids_same_type:
            all_relations.pop(rid, None)

        # after rule 1, if <2 remaining or only 1 type, skip LLM
        kept_rels = [r for r in kept_rels if r.get("rid") in all_relations]
        # print(kept_rels)
        types_left = {(r.get("relation_type") or r.get("type") or "").strip() for r in kept_rels}
        types_left.discard("")
        if len(kept_rels) < 2 or len(types_left) < 2:
            continue

        # -----------------------
        # Rule 2: different types -> ask LLM
        # -----------------------
        relations = json.dumps(
            [
                {k: v for k, v in r.items()}  # 可选：去掉 conf 等噪声字段
                for r in kept_rels
            ],
            ensure_ascii=False,
            indent=2,
        )

        raw = extractor.dedup_relations(
            # text=content,
            relations=relations,
        )

        try:
            out = json.loads(raw)
            print(out)
        except Exception:
            # safest fallback: keep
            continue

        decision = (out.get("decision") or "").strip().lower()
        output = out.get("output") or {}

        if decision != "drop":
            continue

        drop_types = output.get("drop_relation_types") or []
        if not isinstance(drop_types, list) or not drop_types:
            continue

        # Safety: only allow dropping types that appear in this pair
        drop_types = [t for t in drop_types if isinstance(t, str) and t.strip() in types_left]
        if not drop_types:
            continue

        drop_set = set(drop_types)
        for r in kept_rels:
            rid = r.get("rid")
            if not rid:
                continue
            rtype = (r.get("relation_type") or r.get("type") or "").strip()
            if rtype in drop_set:
                all_relations.pop(rid, None)

    return all_relations


In [ ]:
all_relations = dedup_multi_relations(
    content=cleaned,       # 场景文本
    all_relations=all_relations,
)

[{'subject': 'St Anne', 'object': 'F.B.I.', 'relation_type': 'hostility_with', 'relation_name': 'is critical of', 'persistence': 'momentary', 'description': 'St ANNE makes a sarcastic remark about the F.B.I. being notorious for withholding information, indicating a momentary antagonistic attitude.', 'rid': 'social_relations#1', 'conf': 0.8}, {'subject': 'St Anne', 'object': 'F.B.I.', 'relation_type': 'subordinate_to', 'relation_name': 'is not under the command of', 'persistence': 'phase', 'description': "St ANNE's sarcastic comment about the F.B.I. withholding information suggests a professional distance or lack of subordination, implying his operation is independent.", 'rid': 'semantic_relations#8', 'conf': 0.8}]
{'decision': 'keep', 'output': {}}


In [ ]:
len(list(all_relations.items()))

54

In [ ]:
len(entity_extraction)

20

In [ ]:
from __future__ import annotations

import json
from collections import defaultdict
from typing import Any, Dict, List, Tuple, Optional


def load_relation_basic_info(path: str) -> List[Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError(f"Expected a List[dict] in {path}, got: {type(data)}")
    return data


def find_multi_edges(
    relations: List[Dict],
    *,
    exclude_same_chapter: bool = True,
) -> Dict[Tuple[str, str], List[Dict]]:
    """
    多条边（不区分 relation_type）：
    (subject_id, object_id)

    Args:
        exclude_same_chapter:
            True  -> 过滤掉所有关系都来自同一个 base_chapter 的情况
            False -> 不做该过滤
    """
    buckets = defaultdict(list)

    for r in relations:
        sid = r.get("subject_id")
        oid = r.get("object_id")

        if not sid or not oid:
            continue

        key = (sid, oid)
        buckets[key].append(r)

    out = {}

    for key, rels in buckets.items():
        if len(rels) <= 1:
            continue

        if exclude_same_chapter:
            chapters = {
                r.get("base_chapter")
                for r in rels
                if r.get("base_chapter") is not None
            }

            # 全部来自同一个 chapter -> 跳过
            if len(chapters) <= 1:
                continue

        out[key] = rels

    return out

def print_multi_edges(
    multi_edges: Dict[Tuple[str, str], List[Dict]],
    *,
    max_groups: int = 20,
    max_edges_per_group: int = 10,
) -> None:
    print(f"Found {len(multi_edges)} multi-edge groups")
    print("=" * 100)

    for i, ((sid, oid), rels) in enumerate(multi_edges.items()):
        if i >= max_groups:
            break

        first = rels[0]

        subj_name = first.get("subject", "<UNK>")
        obj_name = first.get("object", "<UNK>")

        print(f"[{i+1}] {subj_name} ({sid})  --->  {obj_name} ({oid})")
        print(f"    total_edges = {len(rels)}")

        # chapter 分布
        chapter_counts = {}
        for r in rels:
            ch = r.get("base_chapter", "<UNK>")
            chapter_counts[ch] = chapter_counts.get(ch, 0) + 1

        # print("    chapters:")
        # for ch, cnt in sorted(chapter_counts.items(), key=lambda x: (-x[1], x[0])):
        #     print(f"      - {ch}: {cnt}")

        # 具体边
        print("    edges:")
        for j, r in enumerate(rels):
            if j >= max_edges_per_group:
                break

            print(
                f"      - {r.get('relation_type')} | "
                f"base={r.get('base_chapter')} | "
                f"chapter={r.get('chapter_key')} | "
                f"chunks={len(r.get('chunk_ids', []))} | "
                f"rid={r.get('id')}"
            )

        if len(rels) > max_edges_per_group:
            print(f"      ... {len(rels) - max_edges_per_group} more")

        print("-" * 100)



In [ ]:
relations = load_relation_basic_info("data/knowledge_graph/relation_info_refined.json")

# 1) 完全重复的边
strict_multi = find_multi_edges(relations)
print("=== STRICT MULTI-EDGES ===")
print_multi_edges(strict_multi)



=== STRICT MULTI-EDGES ===
Found 20 multi-edge groups
[1] Ross (ent_5fbae50a736c)  --->  Berlin (ent_2cfa50f104b1)
    total_edges = 5
    edges:
      - speaks_to | base=scene_11 | chapter=scene_11_part_1 | chunks=0 | rid=rel_4aa69206d7ac
      - favorable_toward | base=scene_11 | chapter=scene_11_part_1 | chunks=0 | rid=rel_66e5c0c3d073
      - affinity_with | base=scene_12 | chapter=scene_12_part_1 | chunks=0 | rid=rel_e375857beee0
      - unfavorable_toward | base=scene_26 | chapter=scene_26_part_1 | chunks=0 | rid=rel_551fb21aecb7
      - hostility_with | base=scene_34 | chapter=scene_34_part_1 | chunks=0 | rid=rel_4832ea98a8b6
----------------------------------------------------------------------------------------------------
[2] Berlin (ent_2cfa50f104b1)  --->  Ross (ent_5fbae50a736c)
    total_edges = 5
    edges:
      - speaks_to | base=scene_11 | chapter=scene_11_part_1 | chunks=0 | rid=rel_7ea082b2c74c
      - affinity_with | base=scene_11 | chapter=scene_11_part_2 | chunks

In [ ]:
x = "Bare wooden boards and the sound of singing birds. This house hasn't been lived in for years. No furniture other than a new mattress in the middle of the floor. Still in polythene wraps. BERLIN just about awake on top of it. Ten seconds of disorien- tation while he puts this together. A stone fireplace. Stairs leading to what's got to be a tiny room above. With enough ef- fort this place could be charming. But right now it's a wreck."

In [ ]:
import json
from typing import Dict, Any, List

def load_global_entities_sorted_by_degree(path: str) -> List[Dict[str, Any]]:
    """
    Load entities from JSON file, filter scope == 'global',
    and sort by total_degree descending.
    """
    with open(path, "r", encoding="utf-8") as f:
        data: Dict[str, Dict[str, Any]] = json.load(f)

    global_entities = [
        ent for ent in data.values()
        if ent.get("scope") == "global"
    ]

    global_entities_sorted = sorted(
        global_entities,
        key=lambda x: x.get("total_degree", 0),
        reverse=True
    )

    return global_entities_sorted


In [ ]:
entities = load_global_entities_sorted_by_degree("data/knowledge_graph/entity_info_refined.json")

In [ ]:
entities[0]["source_chapters"]

['scene_100_part_1',
 'scene_101_part_1',
 'scene_102_part_1',
 'scene_104_part_1',
 'scene_105_part_1',
 'scene_106_part_1',
 'scene_107_part_1',
 'scene_108_part_1',
 'scene_109_part_1',
 'scene_110_part_1',
 'scene_111_part_1',
 'scene_112_part_1',
 'scene_115_part_1',
 'scene_118_part_1',
 'scene_11_part_1',
 'scene_11_part_2',
 'scene_120_part_1',
 'scene_121_part_1',
 'scene_122_part_1',
 'scene_123_part_1',
 'scene_124_part_1',
 'scene_126_part_1',
 'scene_126_part_2',
 'scene_128_part_1',
 'scene_128_part_2',
 'scene_129_part_1',
 'scene_12_part_1',
 'scene_130_part_1',
 'scene_133_part_1',
 'scene_136_part_1',
 'scene_136_part_2',
 'scene_138_part_1',
 'scene_13_part_1',
 'scene_142_part_1',
 'scene_143_part_1',
 'scene_144_part_1',
 'scene_145_part_1',
 'scene_146_part_1',
 'scene_147_part_1',
 'scene_147_part_2',
 'scene_148_part_1',
 'scene_149_part_1',
 'scene_14_part_1',
 'scene_150_part_1',
 'scene_150_part_2',
 'scene_151_part_1',
 'scene_152_part_1',
 'scene_155_part_1

In [ ]:
selected_entities = [ent for ent in entities if ent["total_degree"] >= 2]

In [ ]:
len(selected_entities)

137

In [ ]:
with open("data/knowledge_graph/entity_info_refined.json", "r") as f:
    data = json.load(f)

In [ ]:
len(data)

3090

In [ ]:
1+1

2